# Episode 02: When does LeBron first reach his season high?

We estimate an empirical PMF and CDF for the **team game number of the first occurrence of each season's highest-scoring game**. We then measure age relationships and test an OLS prediction using age plus debut total points.

## 1. Set up the independent episode folder

The notebook works when Jupyter starts from the repository root, this episode folder, or its `python` folder. `fetch_data.py` is intentionally separate: it downloads official NBA responses, preserves source URLs, and verifies detailed logs against season totals.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
if cwd.name == 'python':
    EPISODE_ROOT = cwd.parent
elif (cwd / 'data' / 'season_summary.csv').exists():
    EPISODE_ROOT = cwd
else:
    EPISODE_ROOT = cwd / 'episodes' / 'episode-02-season-high'

sys.path.insert(0, str(EPISODE_ROOT / 'python'))
from episode_02_season_high import (
    correlation_table, empirical_distribution, evaluate, fit_ols,
    load_data, predict, walk_forward
)

## 2. Load and verify 23 season-level observations

One row represents one completed regular season, from 2003–04 through 2025–26. `team_game_number` counts team games, even when LeBron did not appear. If the season high is tied, the earliest occurrence wins.

In [2]:
seasons = load_data()
columns = [
    'season', 'team_games', 'player_games', 'age_at_debut',
    'debut_fgm', 'debut_points', 'season_high_points',
    'first_high_team_game_number', 'first_high_player_game_number'
]
display(seasons[columns])
print('Rows:', len(seasons))
print('82-game seasons:', seasons['is_82_game_season'].sum())

,season,team_games,player_games,age_at_debut,debut_fgm,debut_points,season_high_points,first_high_team_game_number,first_high_player_game_number
0,2003-04,82,79,18.828586,12,25,41,72,69
1,2004-05,82,80,19.844350,8,28,56,64,62
2,2005-06,82,79,20.840948,9,31,52,19,19
3,2006-07,82,78,21.837546,11,26,41,61,59
4,2007-08,82,75,22.834144,2,10,51,38,33
5,2008-09,82,81,23.828005,9,22,55,53,53
6,2009-10,82,76,24.824603,12,38,48,34,34
7,2010-11,82,79,25.821201,10,31,51,49,47
8,2011-12,66,62,26.984811,11,37,41,52,51
9,2012-13,82,76,27.833563,10,26,40,55,55


Rows: 23
82-game seasons: 20


## 3. Empirical PMF, CDF and density

The target is an integer game number, so exact probabilities use a **PMF**. The CDF answers: “By team game $k$, what fraction of historical first season highs had occurred?” A histogram of normalized season position can approximate a density, but its bar height is not $P(G=k)$.

In [3]:
distribution = empirical_distribution(seasons['first_high_team_game_number'])
display(distribution.loc[[10, 20, 30, 41, 49, 60, 70, 82]])
print('PMF sums to:', distribution['pmf'].sum())
print('CDF at game 41:', f"{distribution.loc[41, 'cdf']:.1%}")

,pmf,cdf
game_number,,
10,0.043478,0.086957
20,0.000000,0.217391
30,0.043478,0.347826
41,0.000000,0.434783
49,0.086957,0.565217
60,0.000000,0.826087
70,0.000000,0.956522
82,0.000000,1.000000


PMF sums to: 1.0
CDF at game 41: 43.5%


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(distribution.index, distribution.pmf, color='steelblue')
axes[0].set(title='Empirical PMF', xlabel='First season-high team game #', ylabel='Relative frequency')
axes[1].step([0, *distribution.index], [0, *distribution.cdf], where='post', color='darkorange')
axes[1].set(title='Empirical CDF', xlabel='Team game # k', ylabel='P(G ≤ k)', ylim=(0, 1.05))
plt.tight_layout(); plt.show()

/var/folders/f3/1ylyl6_s12xgw6cshdx6xg8r0000gn/T/ipykernel_61688/2720155386.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Age correlations

Age is measured at LeBron's first appearance that season. `debut_fgm` is field goals made. `season_high_points` is the maximum single-game total points in that completed regular season. Pearson describes linear association; Spearman describes rank association. Neither establishes a causal age effect.

In [5]:
correlations = correlation_table(seasons)
display(correlations)

,x,y,n,pearson_r,spearman_rho
0,age_at_debut,debut_fgm,23,-0.085498,-0.095314
1,age_at_debut,season_high_points,23,-0.257817,-0.267793


## 5. Linear regression

Fit $\hat G = b_0 + b_1 \times age + b_2 \times debut\ points$. These predictors are available only after the season debut, so this is an after-debut forecast.

In [6]:
model = fit_ols(seasons)
model

{'target': 'first_high_team_game_number',
 'n': 23,
 'features': ['age_at_debut', 'debut_points'],
 'intercept': 75.72797257063505,
 'age_coefficient': -1.017630289734481,
 'debut_points_coefficient': -0.15421467325456995,
 'training_r2': 0.1207903792333046,
 'training_mae': 15.613659483782708}

## 6. Chronological evaluation

Training fit is optimistic. For an honest historical check, train on the first ten seasons, predict the next season, expand the window, and repeat. Compare the model against the mean and median of earlier target values.

In [7]:
predictions = walk_forward(seasons)
display(predictions)
evaluate(predictions)

,test_season,train_through,train_n,actual,prediction,past_mean,past_median
0,2013-14,2012-13,10,57.0,46.986389,49.700000,52.5
1,2014-15,2013-14,11,59.0,51.269782,50.363636,53.0
2,2015-16,2014-15,12,10.0,50.953637,51.083333,54.0
3,2016-17,2015-16,13,22.0,40.896015,47.923077,53.0
4,2017-18,2016-17,14,9.0,30.458380,46.071429,52.5
5,2018-19,2017-18,15,16.0,23.541483,43.600000,52.0
6,2019-20,2018-19,16,56.0,23.095279,41.875000,50.5
7,2020-21,2019-20,17,18.0,25.914046,42.705882,52.0
8,2021-22,2020-21,18,63.0,14.265241,41.333333,50.5
9,2022-23,2021-22,19,44.0,27.678753,42.473684,52.0


{'n_test_seasons': 13,
 'prediction': {'mae_games': 18.43034865525386,
  'rmse_games': 23.05679858930157},
 'past_mean': {'mae_games': 18.80833950720233,
  'rmse_games': 22.093980650641647},
 'past_median': {'mae_games': 20.576923076923077,
  'rmse_games': 25.498491659764444}}

## 7. Try a hypothetical input

The inputs below are a scenario, not observed 2026–27 data. Keep the raw OLS output before rounding; linear regression does not enforce the valid range of games 1 through 82.

In [8]:
age = 41.8
debut_points = 25
raw_prediction = predict(model, age, debut_points)
print(f'Raw prediction: {raw_prediction:.2f}')
print(f'Nearest team game: {round(raw_prediction)}')

Raw prediction: 29.34
Nearest team game: 29


## 8. Interpret the result

The fitted equation exists, but predictive usefulness is weak: the expanding-window MAE is about 18 games and only slightly better than the historical-mean baseline. Age and debut points leave most of the timing variation unexplained. See `data/results.json` and the four generated figures for the exact results and sensitivity analyses.